In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv("../Raw-Data/Enron.csv")
print(df.columns)
print(df.shape)
df.head()

Index(['subject', 'body', 'label'], dtype='object')
(29767, 3)


,subject,body,label
0,"hpl nom for may 25 , 2001",( see attached file : hplno 525 . xls )\r\n- h...,0
1,re : nom / actual vols for 24 th,- - - - - - - - - - - - - - - - - - - - - - fo...,0
2,"enron actuals for march 30 - april 1 , 201","estimated actuals\r\nmarch 30 , 2001\r\nno flo...",0
3,"hpl nom for may 30 , 2001",( see attached file : hplno 530 . xls )\r\n- h...,0
4,"hpl nom for june 1 , 2001",( see attached file : hplno 601 . xls )\r\n- h...,0


In [4]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\r", " ").replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["subject_clean"] = df["subject"].apply(clean_text)
df["body_clean"] = df["body"].apply(clean_text)

df["email_text"] = (
    df["subject_clean"] + " " + df["body_clean"]
        ).str.strip()
df["label"] = pd.to_numeric(df["label"], errors="coerce")


In [5]:
print(df["label"].value_counts(dropna=False))
print(df[["subject", "body", "email_text", "label"]].head())

label
0    15791
1    13976
Name: count, dtype: int64
                                      subject  \
0                   hpl nom for may 25 , 2001   
1            re : nom / actual vols for 24 th   
2  enron actuals for march 30 - april 1 , 201   
3                   hpl nom for may 30 , 2001   
4                   hpl nom for june 1 , 2001   

                                                body  \
0  ( see attached file : hplno 525 . xls )\r\n- h...   
1  - - - - - - - - - - - - - - - - - - - - - - fo...   
2  estimated actuals\r\nmarch 30 , 2001\r\nno flo...   
3  ( see attached file : hplno 530 . xls )\r\n- h...   
4  ( see attached file : hplno 601 . xls )\r\n- h...   

                                          email_text  label  
0  hpl nom for may 25 , 2001 ( see attached file ...      0  
1  re : nom / actual vols for 24 th - - - - - - -...      0  
2  enron actuals for march 30 - april 1 , 201 est...      0  
3  hpl nom for may 30 , 2001 ( see attached file ...      0  
4  h

In [8]:
clean_df = df[["email_text", "label"]].copy()

clean_df = clean_df.dropna(subset=["email_text", "label"])
clean_df = clean_df[clean_df["email_text"].str.len() > 0]
clean_df = clean_df.drop_duplicates(subset=["email_text"]).reset_index(drop=True)

clean_df["label"] = clean_df["label"].astype(int)

print("Cleaned shape:", clean_df.shape)
print(clean_df["label"].value_counts())
clean_df.head()

Cleaned shape: (29423, 2)
label
0    15474
1    13949
Name: count, dtype: int64


,email_text,label
0,"hpl nom for may 25 , 2001 ( see attached file ...",0
1,re : nom / actual vols for 24 th - - - - - - -...,0
2,"enron actuals for march 30 - april 1 , 201 est...",0
3,"hpl nom for may 30 , 2001 ( see attached file ...",0
4,"hpl nom for june 1 , 2001 ( see attached file ...",0


In [11]:
clean_df["email_text"] = clean_df["email_text"].astype(str)

clean_df["email_text"].str.contains(
    r"\b(?:spam|ham|phishing|legitimate)\b",
    case=False,
    na=False
).mean()

np.float64(0.020528158243550962)

In [12]:
clean_df.to_csv("../Processed-Data/enron_cleaned.csv", index=False)
print("Enron cleaned and dataset saved.")

Enron cleaned and dataset saved.
